# Deep Learning in Audio Applications
### Teach machines to listen: your own audio-to-text notebook

You will take speech, turn it into text, see where that breaks, build the two ideas that let a computer read a sequence
(an **RNN**, then an **LSTM**), and end with **meeting minutes** from a conversation with two speakers.

Run the cells from top to bottom. Cells marked **YOUR TURN** have one line for you to write; a check tells you straight away if it is right.

> **If Colab restarts, or you jump ahead to a later section:** run cell 2 (Setup) again first, then
> every cell above the one you want, in order. Several cells further down reuse things an earlier
> cell set up (`transcript`, `turns`, the trained model) and will show a `NameError` without them.

| section | content | agenda block(s) | minutes |
|---|---|---|---|
| 1 | Audio to text with Python | 2 (part) | ~6 |
| 2 | Where transcription struggles | 2 (part) | ~4 |
| 3 | RNNs: reading words as a sequence (+ Hands-on A, + why memory fades) | 3, 4, 5 | 15 + 10 + 7 |
| 4 | LSTMs: an eraser you can set (+ teaching it, + Hands-on B) | 6, 7, 8 | 20 + 10 + 20 |
| 5 | Meeting minutes (+ your own meeting) | 9 | 15 |


## 0. Setup (run once, about a minute)
On Colab: **Runtime > Change runtime type > T4 GPU** makes Whisper several times faster (the free CPU works too, just slower).
Upload `workshop_files.zip` from the facilitators to the Files panel on the left before running this cell.

In [ ]:
import os, sys, shutil, subprocess, importlib.util, zipfile
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")                  # avoids a library clash on some Windows installs
import warnings; warnings.filterwarnings("ignore", message="FP16")  # Whisper on CPU only

if importlib.util.find_spec("whisper") is None:                       # Whisper: speech-to-text from OpenAI
    print("Installing Whisper, about a minute...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai-whisper"], check=True)
if os.path.exists("workshop_files.zip") and not os.path.exists("prototype"):
    zipfile.ZipFile("workshop_files.zip").extractall(".")              # the code and audio for this workshop
PROTO = next((p for p in ("prototype", "../prototype") if os.path.isdir(p)), None)   # ../prototype when run from the repo
PROTO or sys.exit("Upload workshop_files.zip to the Files panel (left), then run this cell again.")
sys.path.insert(0, PROTO)
print("ffmpeg (reads audio files):", "found" if shutil.which("ffmpeg") else "NOT FOUND - install it (Colab has it already)")
print("ready")

## 1. Audio to text with Python
*Facilitator-run, during the talk -- sections 1-2 together are Agenda block 2, about 10 minutes*
*(roughly 6 min here, 4 min in section 2). Participants: run these cells too (about a minute on*
*a T4); the later sections reuse their results. Nothing in either section is YOUR TURN.*

Whisper is a ready-made speech-to-text model. Three lines: load it, give it a file, read the text.
Bigger models hear more but run slower: `tiny.en`, `base.en`, `small.en`, `medium.en` (the `.en` ones know only English). We pick a size that fits your machine.


In [ ]:
import torch, whisper

MODEL = "small.en" if torch.cuda.is_available() else "base.en"        # a GPU can afford the bigger model
print("using", MODEL, "on", "the GPU" if torch.cuda.is_available() else "the CPU", "(about 20s on CPU)")
model = whisper.load_model(MODEL)
result = model.transcribe("audio/sample_meeting.wav", language="en")
print(result["text"])


### What Whisper is reading
A spectrogram is like sheet music for speech: **time runs left to right, pitch runs bottom to top**.
Here is the waveform and the spectrogram for a short clip: three sentences from the meeting.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

wave_samples = whisper.load_audio("audio/sample_short.wav")
t = np.arange(len(wave_samples)) / 16000

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax1.plot(t, wave_samples, linewidth=0.5)
ax1.set_ylabel("amplitude")
ax1.set_title("Waveform")
with np.errstate(divide="ignore"):      # a silent stretch gives log10(0); matplotlib handles it, just don't warn
    ax2.specgram(wave_samples, Fs=16000, cmap="magma")
ax2.set_ylabel("pitch (Hz)")
ax2.set_xlabel("time (s)")
ax2.set_title("Spectrogram -- time left to right, pitch bottom to top")
plt.tight_layout()
plt.show()


**Order matters.** Play a short clip forwards, then backwards -- same sounds, but backwards it
stops being words. Whisper looks at the spectrogram in time order too -- it's a different kind
of model (it weighs a whole 30-second window at once, not one step at a time like the RNN
below), but order still matters to it: reverse the clip and the "words" it would read make no
sense either.


In [ ]:
from IPython.display import Audio, display

clip = wave_samples[:8000]          # about half a second -- roughly one word
print("forwards:")
display(Audio(clip, rate=16000))
print("backwards:")
display(Audio(clip[::-1], rate=16000))


**What Whisper-style models actually see.** Not the waveform directly -- a short window every
10 ms, turned into a handful of numbers (an MFCC frame, the classic speech-model feature; Whisper
itself uses a similar but larger spectrogram feature). This is the picture above, in the form a
model reads.


In [ ]:
from audio_features import mfcc

samples = whisper.load_audio("audio/sample_short.wav")
frames = mfcc(samples)
print(f"{len(samples) / 16000:.0f} seconds of audio  ->  {frames.shape[0]} frames x {frames.shape[1]} numbers (one frame every 10 ms)")


Our pipeline wants more than one block of text: it wants **who said what**. `get_transcript` listens for pauses and
starts a new turn each time one person stops and the next starts (here: a pause longer than 1.5 s).

In [ ]:
os.environ["WHISPER_MODEL"] = MODEL          # the pipeline's transcriber reads this
from transcriber import get_transcript

print("transcribing again with speaker splitting (about 20s on CPU)...")
transcript = get_transcript("audio/sample_meeting.wav", pause_gap=1.5)
print(transcript)


## 2. Where transcription struggles
*Facilitator-run, about 4 of block 2's 10 minutes (see section 1's note).*

Real recordings are not clean. The three things that hurt most: **background noise**, **a poor microphone**, and **accents**.
We score by how many of the words Whisper gets right (`word_accuracy`: 1.0 = every word).


In [ ]:
import numpy as np
from audio_check import word_accuracy

SAID = ("Venue is confirmed for the 14th, catering still pending. "
        "Can you follow up with catering by Friday? Yes, I'll email them today.")
clean = whisper.load_audio("audio/sample_short.wav")          # the same three sentences, 14 seconds

def hear(samples):
    return model.transcribe(samples, language="en")["text"].strip()

print("clean      ", f"{word_accuracy(SAID, hear(clean)):.0%}", "|", hear(clean), "  (about 5s on CPU)")


**Noise.** Add random hiss to the recording. How loud can it get before Whisper breaks? (try 0.02, 0.05, 0.1, 0.2)

In [ ]:
NOISE = 0.1
noisy = (clean + NOISE * np.random.default_rng(0).normal(size=clean.shape)).astype("float32")
heard = hear(noisy)
print(f"noise {NOISE}:", f"{word_accuracy(SAID, heard):.0%}", "|", heard)

**A poor microphone.** Keep only every 4th sample, like a cheap mic or a phone line. The high sounds (s, t, f) disappear.

In [ ]:
poor = np.repeat(clean[::4], 4).astype("float32")
heard = hear(poor)
print("poor mic  :", f"{word_accuracy(SAID, heard):.0%}", "|", heard)

**Accents.** Record yourself on your phone reading the same three sentences, with about 2 seconds between them:

> Venue is confirmed for the 14th, catering still pending. / Can you follow up with catering by Friday? / Yes, I'll email them today.

Upload it to the Files panel, put its name below, and compare with the clean voice. What did Whisper get wrong, and why?

In [ ]:
MY_VOICE = None            # for example "my_voice.m4a"
if MY_VOICE:
    mine = whisper.load_audio(MY_VOICE)
    heard = hear(mine)
    print("my voice  :", f"{word_accuracy(SAID, heard):.0%}", "|", heard)
else:
    print("Set MY_VOICE to a file you uploaded, then run this cell again.")

**Kahoot: which gives the most accurate transcript?** Four clips of the exact same sentence --
an ideal recording and three noisy versions. The facilitator plays each one and the room votes on
the ranking, best to worst, before the answer key cell below is run.


In [ ]:
NOISY_CLIPS = [
    ("1. ideal", "noisy_1_ideal.wav"),
    ("2. background hiss", "noisy_2_hiss.wav"),
    ("3. poor microphone", "noisy_3_poor_mic.wav"),
    ("4. a second voice underneath", "noisy_4_second_voice.wav"),
]

print("Vote now: which order, best to worst, before running the next cell?")
print()
for label, filename in NOISY_CLIPS:
    print(label)
    display(Audio(f"audio/{filename}"))


**Answer key.** Measured live, same as every other number in this notebook -- not assumed.
The percentages depend strongly on the Whisper model size -- `MODEL` in section 1 (`base.en`,
`small.en` or `medium.en`), not a fixed property of the clips. All three rows below are CPU runs
(no GPU available when this was built):

| model | ideal | second voice | hiss | poor mic |
|---|---|---|---|---|
| `base.en` | 100% | 61% | 52% | 35% |
| `small.en` (this notebook's GPU default) | 100% | 87% | 83% | 22% |
| `medium.en` | 100% | 91% | 83% | 35% |

This order -- ideal, then the second voice underneath, then hiss, then the poor microphone --
held on the three sizes we tried. The exact percentages are only for this machine's run; run the
cell below to get yours.


In [ ]:
print("running Whisper on all four clips, about 20s total on CPU...")
for label, filename in NOISY_CLIPS:
    heard = hear(whisper.load_audio(f"audio/{filename}"))
    print(f"{label:32s} {word_accuracy(SAID, heard):.0%}  | {heard}")


### 2b. Say it yourself (live mic) *(optional, time permitting -- about 1 minute)*
**Predict:** if real background noise gets mixed into your recording, how much do you think
Whisper's accuracy will drop -- a little, or a lot?

In [ ]:
import live_mic

SAID_LIVE, clear_live_path, used_fallback = live_mic.say("block2_clear", model, seconds=5)
print("heard:", SAID_LIVE)

clear_live = whisper.load_audio(clear_live_path)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
t = np.arange(len(clear_live)) / 16000
ax1.plot(t, clear_live, linewidth=0.5)
ax1.set_ylabel("amplitude")
ax1.set_title("Your waveform")
with np.errstate(divide="ignore"):
    ax2.specgram(clear_live, Fs=16000, cmap="magma")
ax2.set_ylabel("pitch (Hz)")
ax2.set_xlabel("time (s)")
ax2.set_title("Your spectrogram")
plt.tight_layout()
plt.show()

**Try it:** say the same sentence again, this time with real background noise -- tap the
desk, rustle paper, or have music playing nearby.

In [ ]:
from audio_check import word_accuracy

NOISY_LIVE, noisy_live_path, _ = live_mic.say("block2_noisy", model, seconds=5)
print("heard:", NOISY_LIVE)
print(f"accuracy vs your own clear recording: {word_accuracy(SAID_LIVE, NOISY_LIVE):.0%}")

**Why:** the exact mix of sounds you just made is what Whisper reads -- add anything else to
it (noise, a bad mic, an accent it hasn't heard much of) and accuracy drops, same as cells 15-18
showed on the sample recording. This is YOUR voice and YOUR noise, not a canned example.

**Try it:** record one word on its own, then hear it played backwards.

In [ ]:
WORD_LIVE, word_live_path, _ = live_mic.say("block2_backward", model, seconds=3)
print("heard:", WORD_LIVE)

word_samples = whisper.load_audio(word_live_path)
print("forwards:")
display(Audio(word_samples, rate=16000))
print("backwards:")
display(Audio(word_samples[::-1].copy(), rate=16000))
backward_heard = model.transcribe(word_samples[::-1].copy(), language="en")["text"].strip()
print("Whisper on the reversed clip:", repr(backward_heard) if backward_heard else "(nothing)")

**Why:** order matters to Whisper (and to the RNN two sections below) -- reversed, your own
word stops being a word Whisper recognises, same as the forwards/backwards clip in section 1.

## 3. RNNs: reading words as a sequence (Agenda block 3, 15 min)
When you read a sentence, you don't reread every word from scratch -- you carry a running summary
in your head and update it with each new word. An RNN does the same: it reads one **word** at a
time and carries a small "summary" (its memory) forward:

`new memory = mix of (this word) + (old memory)`, written as `h = tanh(W_x·x + W_h·h_prev + b)`

(Whisper looks at its spectrogram in time order too, though it's a different kind of
model (see section 1) -- the same "order matters" idea as the reversed word, just for
sound instead of words.)
reversed word in section 1, just for sound instead of words.)


In [ ]:
import numpy as np
from lstm_model import model_words

sentence = "Can you follow up with catering by Friday?"
words = model_words(sentence)
vocab = {w: i for i, w in enumerate(sorted(set(words)))}
numbers = [vocab[w] for w in words]

print("words  :", words)
print("numbers:", numbers, " (each word -> its own number, via a vocabulary)")

# Step a tiny RNN over these words, one at a time, to show "reads one word at a time" --
# toy random weights and embeddings, not the trained model (that comes in section 4).
rng = np.random.default_rng(0)
dim = 4
toy_embeddings = rng.normal(size=(len(vocab), dim))
W_x_toy, W_h_toy, b_toy = rng.normal(scale=0.5, size=(dim, dim)), rng.normal(scale=0.5, size=(dim, dim)), np.zeros(dim)

print()
print("Stepping the RNN one word at a time:")
memory = np.zeros(dim)
for word, number in zip(words, numbers):
    x = toy_embeddings[number]
    memory = np.tanh(W_x_toy @ x + W_h_toy @ memory + b_toy)
    print(f"  after {word!r:10s} memory = {np.round(memory, 2)}")


### 3b. Watch it read YOUR sentence (live mic)
**Predict:** will the memory update at every single word you say, or only some of them?

**Try it:** say any sentence, 5-15 words, at a normal pace.

In [ ]:
toy_live_rng = np.random.default_rng(2)
live_hidden = 8
W_x_live = toy_live_rng.normal(0, 0.5, (live_hidden, 8))
W_h_live = toy_live_rng.normal(0, 0.25, (live_hidden, live_hidden))
b_live = np.zeros(live_hidden)

def toy_rnn_step(x, h_prev):
    return np.tanh(W_x_live @ x + W_h_live @ h_prev + b_live)

SENTENCE_LIVE, sentence_live_path, _ = live_mic.say("block3_sentence", model, seconds=5)
print("heard:", SENTENCE_LIVE)
live_words = model_words(SENTENCE_LIVE)

mem_trace = []
h = np.zeros(live_hidden)
for w in live_words:
    h = toy_rnn_step(live_mic.word_vector(w), h)
    mem_trace.append(np.abs(h).mean())

plt.figure(figsize=(8, 3))
plt.bar(range(len(live_words)), mem_trace)
plt.xticks(range(len(live_words)), live_words, rotation=45, ha="right")
plt.ylabel("memory (mean |h|)")
plt.title("The toy RNN's memory, updating one of YOUR words at a time")
plt.tight_layout()
plt.show()
print(f"The memory changed at every one of your {len(live_words)} words -- never flat, whatever you said.")

**Why:** this is the same toy RNN as Part A below, fed through a fixed hash-seeded vector per
word (`live_mic.word_vector` -- not a learned embedding; the same word always gets the same
vector, but nothing here has been trained) instead of the scripted sentence. It updates at every
word, whatever you actually said -- that's what "reads one word at a time" means in practice.

### YOUR TURN (Part A): one step of an RNN (Agenda block 4, Hands-on A, 10 min)
At every step the RNN mixes **the new input** with **its old memory**:

`new memory = tanh( W_x × x  +  W_h × old memory  +  b )`

In numpy `×` is `@`, and tanh is `np.tanh`. Write that one line.

*`x` = this word, as numbers. `h_prev` = the memory so far. `W_x`, `W_h` = numbers the model*
*has learned (how much each one matters). `b` = a small nudge.*


In [ ]:
import numpy as np, hands_on

def rnn_step(x, h_prev, W_x, W_h, b):
    # TODO: return the new memory (the line above, in numpy)
    return None

hands_on.rnn_step = rnn_step
hands_on.MEMORY_ON = True
hands_on.part_a()

<details><summary>Stuck? (click to expand)</summary>

Verified by actually making each mistake and running it:

- **A 2x2 grid instead of a list of numbers** -- you used `*` somewhere `@` should be
  (`W_h * h_prev` multiplies element by element instead of mixing the memory together).
- **A number outside -1..1** (like `-3.15`) -- `tanh` is missing.
- **Close, but not quite** -- check you added `+ b` at the end.
- **`returned None`** -- the line still ends with `return None`; replace it with your formula.

</details>


Now **break the memory**. Two steps, both needed: delete `+ W_h @ h_prev` from your line above
(not just set a flag -- the term itself has to go), **and** set `hands_on.MEMORY_ON = False` in
that same cell. Run it again. What happens to the first word?

**Ignore the line printed above, "Now set MEMORY_ON = False and run again."** It's only half the
job -- it doesn't mention deleting the term, which is the step that actually matters. Do both
steps above, not just that one.

(If you'd rather see it without editing your own line first, the cell below does exactly this --
an "amnesia" version of `rnn_step`, wired in the same way.)


In [ ]:
# "Amnesia" version: the old-memory term is gone from the line itself, not just flagged off.
# Run this if you want to see the contrast before (or instead of) editing your own cell above.
def amnesia_rnn_step(x, h_prev, W_x, W_h, b):
    return np.tanh(W_x @ x + b)     # no + W_h @ h_prev at all

hands_on.rnn_step = amnesia_rnn_step
hands_on.MEMORY_ON = False
hands_on.part_a()


### 4b. Prove it with your own voice (live mic)
**Predict:** with the memory term in, will "memory on" still tell two sentences apart by the
last word if they only differ in the first word? Will "memory off" (the amnesia version)?

**Try it:** read these two sentences aloud, one per cell below, exactly as written (so they
differ in only the first word):
- "Cats are sitting near the window this evening."
- "Dogs are sitting near the window this evening." 

In [ ]:
CATS_LIVE, _, _ = live_mic.say("block4_cats", model, seconds=5)
print("heard (cats):", CATS_LIVE)

In [ ]:
DOGS_LIVE, _, _ = live_mic.say("block4_dogs", model, seconds=5)
print("heard (dogs):", DOGS_LIVE)

cats_words = model_words(CATS_LIVE)
dogs_words = model_words(DOGS_LIVE)
n_live = min(len(cats_words), len(dogs_words))

def memory_gap(words_a, words_b, use_memory):
    ha = hb = np.zeros(live_hidden)
    gaps = []
    for wa, wb in zip(words_a, words_b):
        xa, xb = live_mic.word_vector(wa), live_mic.word_vector(wb)
        if use_memory:
            ha = np.tanh(W_x_live @ xa + W_h_live @ ha + b_live)
            hb = np.tanh(W_x_live @ xb + W_h_live @ hb + b_live)
        else:
            ha = np.tanh(W_x_live @ xa + b_live)
            hb = np.tanh(W_x_live @ xb + b_live)
        gaps.append(np.abs(ha - hb).sum())
    return gaps

gaps_on = memory_gap(cats_words[:n_live], dogs_words[:n_live], use_memory=True)
gaps_off = memory_gap(cats_words[:n_live], dogs_words[:n_live], use_memory=False)

# Word 1's gap dwarfs the rest on any linear scale (it's just the two words
# "cats"/"dogs" compared with no memory mixed in yet) -- the interesting part is
# what happens AFTER that, so the plot starts at word 2.
plt.figure(figsize=(8, 3))
plt.plot(range(2, n_live + 1), gaps_on[1:], marker="o", label="memory ON")
plt.plot(range(2, n_live + 1), gaps_off[1:], marker="o", label="memory OFF (amnesia)")
plt.xlabel("word (word 1 -- \"cats\"/\"dogs\" itself -- left off; see the printed gap below)")
plt.ylabel("|memory(cats) - memory(dogs)|")
plt.title("From word 2 on: does it still remember your first word was different?")
plt.legend()
plt.tight_layout()
plt.show()
print(f"memory ON:  gap right after the swap (word 2) = {gaps_on[1]:.2f}, still {gaps_on[-1]:.3f} by word {n_live} -- shrinking, but never exactly zero")
print(f"memory OFF: gap right after the swap (word 2) = {gaps_off[1]:.2f}, {gaps_off[-1]:.3f} by word {n_live} -- exactly 0.000 from word 2 on, every time")

**Why:** with the memory term in the line, the gap shrinks but never fully dies -- the model
still carries a trace of your first word. Delete the term (the amnesia version) and the gap
collapses to exactly zero right after the first new word, exactly like Part A's scripted sentence --
now true on sentences you actually said, not just the one in the script.

### Why simple memory fades (Agenda block 5, 7 min)
Think of a game of broken telephone: pass a message through enough people and it's gone. An
RNN's memory fades the same way at every step. This is an **illustration** of that idea, not a
measured property of the trained model (which stacks 5 LSTM layers, not one plain RNN step).

(Part A's toy RNN above kept its own share each step, around 58% on its random weights -- the
dotted curve below is that exact number, measured in this cell, not typed in. The 0.49 curve is
what the untrained LSTM's own gates measured, before training, on this project's actual model.)

This is exactly why **long-range dependencies** matter: if an early line in a meeting sets up
something a much later line depends on, a memory that has already faded can't connect them. The
LSTM section below adds a dial for this -- training decides where to set it.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Part A's toy RNN, replayed here with the correct line so this curve is always right,
# whether or not your own cell above is filled in yet -- same toy setup as Part A (seed 1).
toy_rng = np.random.default_rng(1)
toy_hidden = 8
toy_W_x, toy_W_h, toy_b = (toy_rng.normal(0, 0.5, (toy_hidden, 4)), toy_rng.normal(0, 0.25, (toy_hidden, toy_hidden)), np.zeros(toy_hidden))
toy_words_a = toy_rng.normal(size=(12, 4))
toy_words_b = toy_words_a.copy()
toy_words_b[0] = toy_rng.normal(size=4)
toy_h_a = toy_h_b = np.zeros(toy_hidden)
toy_gaps = []
for x_a, x_b in zip(toy_words_a, toy_words_b):
    toy_h_a = np.tanh(toy_W_x @ x_a + toy_W_h @ toy_h_a + toy_b)
    toy_h_b = np.tanh(toy_W_x @ x_b + toy_W_h @ toy_h_b + toy_b)
    toy_gaps.append(np.abs(toy_h_a - toy_h_b).sum())
toy_kept = np.mean([toy_gaps[k + 1] / toy_gaps[k] for k in range(len(toy_gaps) - 1) if toy_gaps[k] > 1e-12])

plt.figure(figsize=(7, 4))
n = np.arange(48)
for r, note in [(0.49, "untrained setting"), (0.95, "keeping 95%")]:
    retained = r ** n
    plt.plot(n, retained * 100, label=f"r = {r} ({note}): {retained[-1]:.1%} left at n=47")
plt.plot(n, (toy_kept ** n) * 100, linestyle=":", color="gray",
         label=f"Part A's toy RNN above (r = {toy_kept:.2f}, measured here): same fading")
plt.xlabel("steps (n)")
plt.ylabel("% of the original signal retained")
plt.title("Memory retention: r^n (an illustration, not measured)", fontsize=11)
plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8, borderaxespad=0)
plt.ylim(0, 100)
plt.show()

print(f"0.49^47 = {0.49**47:.4%}   0.95^47 = {0.95**47:.1%}   toy RNN: r={toy_kept:.3f}, {toy_kept**47:.4%} left at n=47")


### 5b. Does a SHORT swap survive further than a LONG one? (live mic)
**Predict:** will the first word's effect still be noticeable at the end of a 4-word pair? At
the end of a 15+-word pair?

**Try it:** read these four lines aloud, one per cell below, exactly as written:
- "Red cars drive fast." / "Blue cars drive fast." (4 words, first word swapped)
- "Monday mornings always feel quieter than every other weekday afternoon session we have had
  this term." / the same sentence starting with "Tuesday" instead (16 words, first word
  swapped)

In [ ]:
SHORT_A, _, _ = live_mic.say("block5_short_a", model, seconds=4)
print("heard (short a):", SHORT_A)

In [ ]:
SHORT_B, _, _ = live_mic.say("block5_short_b", model, seconds=4)
print("heard (short b):", SHORT_B)

In [ ]:
LONG_A, _, _ = live_mic.say("block5_long_a", model, seconds=8)
print("heard (long a):", LONG_A)

In [ ]:
LONG_B, _, _ = live_mic.say("block5_long_b", model, seconds=8)
print("heard (long b):", LONG_B)

def fade_curve(words_a, words_b):
    ha = hb = np.zeros(live_hidden)
    gaps = []
    for wa, wb in zip(words_a, words_b):
        ha = toy_rnn_step(live_mic.word_vector(wa), ha)
        hb = toy_rnn_step(live_mic.word_vector(wb), hb)
        gaps.append(np.abs(ha - hb).sum())
    gaps = np.array(gaps)
    return gaps / gaps[0] * 100   # retained %, word 1 = 100%

short_words_a, short_words_b = model_words(SHORT_A), model_words(SHORT_B)
long_words_a, long_words_b = model_words(LONG_A), model_words(LONG_B)
n_short = min(len(short_words_a), len(short_words_b))
n_long = min(len(long_words_a), len(long_words_b))
short_curve = fade_curve(short_words_a[:n_short], short_words_b[:n_short])
long_curve = fade_curve(long_words_a[:n_long], long_words_b[:n_long])

plt.figure(figsize=(8, 4))
n_axis = np.arange(48)
for r, note in [(0.49, "untrained setting"), (0.95, "keeping 95%")]:
    plt.plot(n_axis, (r ** n_axis) * 100, alpha=0.4, label=f"r = {r} ({note})")
plt.plot(range(1, n_short + 1), short_curve, marker="o", color="green", label=f"YOUR short pair ({n_short} words)")
plt.plot(range(1, n_long + 1), long_curve, marker="o", color="red", label=f"YOUR long pair ({n_long} words)")
plt.xlabel("word")
plt.ylabel("% of the first word's effect retained")
plt.title("Your own short vs. long swap, on the same fading curve")
plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8, borderaxespad=0)
plt.ylim(0, 105)
plt.tight_layout()
plt.show()
print(f"short pair: {short_curve[-1]:.0f}% of the first word's effect left by the end")
print(f"long pair:  {long_curve[-1]:.0f}% of the first word's effect left by the end")

**Why:** same toy RNN, same untrained setting as the illustration above -- not the trained
model, which stacks 5 LSTM layers instead of one plain step (see the caveat on the graph above).
Fewer steps leaves more of the first word's effect; by 15+ words it is close to gone. This is the
same r^n fading, just measured on something you actually said.

## 4. LSTMs: an eraser you can set (Agenda block 6, 20 min)
An RNN's memory fades a little at every step, with one fixed fade rate for every word. An LSTM adds a
**dial**: it keeps its memory in a **notebook** (the cell state) with three switches at every
step -- the **eraser** (forget gate, `f`) decides what to rub out, the **pencil** (input gate,
`i`) decides what new thing to write down, and the **voice** (output gate, `o`) decides what to
say out loud right now. Training sets the dial; Hands-on B below lets you turn it yourself.

`new notebook = (what you keep) × (old notebook) + (what you add) × (new note)`, written as
`c = f·c_prev + i·g`

Our model stacks 5 of these layers, like an hourglass: the middle layer looks at four words at
a time (folded together) -- an architecture idea, not something this trained classifier measures
or proves on its own. A `<eot>` token between turns is the "speaker changed" bookmark -- the
model's only signal that someone new is talking.


In [ ]:
from lstm_model import run_pipeline, load_trained_model, tokenize_transcript, pad_to_multiple_of_4

turns, output, turn_ends = run_pipeline(transcript=transcript)
trained = load_trained_model()
vocab = {w: i for i, w in enumerate(trained["vocab"])}
tokens, _ends = tokenize_transcript(turns)
x = trained["embeddings"][[vocab.get(w, 0) for w in tokens]]
_hourglass_out, hourglass_layers = trained["model"].forward(pad_to_multiple_of_4(x))

print("The hourglass: 5 layers, each with its own units and time steps (this notebook's own meeting):")
print(f"  {'layer':<9}{'units':>7}{'time steps':>13}")
for name, out in hourglass_layers.items():
    steps, units = out.shape
    print(f"  {name:<9}{units:>7}{steps:>13}")

layer_names = list(hourglass_layers.keys())
layer_units = [out.shape[1] for out in hourglass_layers.values()]
plt.figure(figsize=(6, 2.5))
plt.bar(layer_names, layer_units)
plt.ylabel("units")
plt.title("The hourglass shape: units per layer")
plt.show()


**Watch the gates.** The eraser, pencil and voice are numbers the model actually computes --
not just names. Layer 1 of the trained model (one ensemble member), replayed by hand from its
shipped weights over the hook's own exchange, "Can you book the room? / Sure.":

*(Caveat: layer 1 only, one of five ensemble members, the mean of its 8 units -- a sketch of
what the gates do, not the whole model.)*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lstm_model import TURN_BOUNDARY_TOKEN, tokenize_transcript

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

gate_example = [("Mei", "Can you book the room?"), ("Ravi", "Sure.")]
gate_tokens, _gate_ends = tokenize_transcript(gate_example)
gate_ids = [vocab.get(w, 0) for w in gate_tokens]
gate_x_seq = trained["embeddings"][gate_ids]
layer1 = trained["model"].layer1

h, c = np.zeros(layer1.hidden_size), np.zeros(layer1.hidden_size)
erasers, pencils, voices = [], [], []
for x in gate_x_seq:
    combined = np.concatenate([h, x])
    f = sigmoid(layer1.W_f @ combined + layer1.b_f)
    i_gate = sigmoid(layer1.W_i @ combined + layer1.b_i)
    c_tilde = np.tanh(layer1.W_c @ combined + layer1.b_c)
    c = f * c + i_gate * c_tilde
    o = sigmoid(layer1.W_o @ combined + layer1.b_o)
    h = o * np.tanh(c)
    erasers.append(f.mean())
    pencils.append(i_gate.mean())
    voices.append(o.mean())

x_pos = np.arange(len(gate_tokens))
width = 0.25
plt.figure(figsize=(9, 3))
plt.bar(x_pos - width, erasers, width, label="eraser (forget)")
plt.bar(x_pos, pencils, width, label="pencil (input)")
plt.bar(x_pos + width, voices, width, label="voice (output)")
plt.xticks(x_pos, gate_tokens, rotation=45, ha="right")
plt.ylabel("gate value (mean of 8 units)")
plt.title("Layer 1's gates, one word at a time (member 0)")
plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8, borderaxespad=0)
plt.tight_layout()
plt.show()

for tok, f_m, i_m, o_m in zip(gate_tokens, erasers, pencils, voices):
    marker = "  <- speaker changed" if tok == TURN_BOUNDARY_TOKEN else ""
    print(f"{tok:10s} eraser={f_m:.2f}  pencil={i_m:.2f}  voice={o_m:.2f}{marker}")


`<eot>` is the only signal the model gets that a new speaker is talking -- there is no speaker
name in the input (`tokenize_transcript` drops it; `prototype/demo.py` only puts the real names
back into the minutes afterwards). Here it is in a 2-turn exchange:


In [ ]:
from lstm_model import TURN_BOUNDARY_TOKEN, tokenize_transcript

example = [("Alex", "Can you follow up with catering by Friday?"), ("Priya", "Yes, I'll email them today.")]
example_tokens, example_ends = tokenize_transcript(example)
print(example_tokens)
print(f"{TURN_BOUNDARY_TOKEN!r} marks \"speaker changed\" between the two turns "
      f"(turn_end_indices = {example_ends}).")


In [ ]:
import numpy as np
from lstm_model import fold_time, unfold_time
x = np.arange(16.0).reshape(8, 2)
print("8 steps of 2 numbers, folded:", fold_time(x).shape, "(4 steps of 4); unfolded back is identical:", np.array_equal(unfold_time(fold_time(x)), x))

### Teaching it, and catching it out (Agenda block 7, 10 min)
Training works like marking practice papers: the model sees a meeting line with the right
label, guesses, is told how wrong it was, and adjusts a little -- repeated over many lines.

`label -> prediction -> loss -> update`

How wrong is measured as `loss = -log(p(the model gave the correct label))` (the natural log):
confident and right is a tiny loss, confident and wrong is a large one. Worked example, on this
notebook's own meeting:


In [ ]:
from lstm_model import label_probabilities
from meeting_minutes import label_turns_stub

gold = label_turns_stub(turns)
probs = label_probabilities(turns, trained)

for (speaker, text), g, p in zip(turns, gold, probs):
    if "email them today" in text or "Venue is confirmed" in text:
        correct = g["category"]
        p_correct = p[trained["labels"].index(correct)]
        print(text)
        print(f"  gold = {correct}, p(correct) = {p_correct:.4f}, loss = {-np.log(p_correct):.2f}")
        print()


The model is confident AND right on "Yes, I'll email them today." (tiny loss), and confident
AND **wrong** on "Venue is confirmed for the 14th..." (a decision stated as a plain fact, no
"agree" or "let's" in it) -- a large loss, exactly the kind of line that pulls training the most.

**Testing it on lines it has never seen.** A simple keyword checklist (does the line contain
"confirmed", "agreed", ...?) versus the trained model, on a held-out meeting -- a student
council, never used in training. This is the team's **best-case** held-out meeting for the
trained model, not a typical one: across all 17 held-out meetings (`holdout_data.py`) the
trained LSTM scores 131/149 (0.879) and 104/122 (0.852) (`python prototype/run_tests.py`
check k) against the keyword checklist's 111/149 (0.745) and 93/122 (0.762) (a per-meeting
count over `holdout_data.py`, not printed by any existing script) -- a real, solid improvement,
but nowhere near 17/18. On 3 of those 17 meetings the keyword checklist actually scores
*better* than the trained model.


In [ ]:
from pipeline import load_text
from meeting_minutes import label_turns_keywords, label_turns_model

held_out_text, held_out_gold = load_text(f"{PROTO}/demo_student_council.txt")
held_out_turns, held_out_output, held_out_ends = run_pipeline(transcript=held_out_text)
kw_labels = label_turns_keywords(held_out_turns)
lstm_labels = label_turns_model(held_out_turns, held_out_output, held_out_ends)

kw_right = sum(k["category"] == g for k, g in zip(kw_labels, held_out_gold))
lstm_right = sum(l["category"] == g for l, g in zip(lstm_labels, held_out_gold))
print(f"keyword rules: {kw_right}/{len(held_out_turns)} turns right")
print(f"trained LSTM:  {lstm_right}/{len(held_out_turns)} turns right")
print()

for (speaker, text), g, l in zip(held_out_turns, held_out_gold, lstm_labels):
    if l["category"] != g:
        print(f"LSTM miss: {text!r}  (gold {g}, got {l['category']})")


The labelling task mostly needs about one previous turn (a bare "Sure." after a request) --
not a long memory. **We don't claim the trained LSTM remembers further back than a plain RNN**
on this task; the long-range-dependency idea from the memory-fade graph above is illustrated by
the toy experiments (Hands-on A and B), not measured by this classifier.

And as seen above: a decision reported as a plain fact ("Venue is confirmed...", "No refreshments
this time...") is where the trained model is weakest -- interestingly, the simple keyword
checklist actually catches "Venue is confirmed" (it contains the word "confirmed"), while the
trained model, which should in principle do better, misses it. Neither approach is perfect; that
is the honest result, not a reason to trust one blindly over the other.


**The "Sure." test -- back to the hook.** The opening exchanges were "Can you book the room?"
"Sure." and "Shall we move it to Friday?" "Sure." -- the room gets both instantly because they
remember the line before. Same two words, run through the model three ways:


In [ ]:
sure_cases = [
    [("Ravi", "Sure.")],
    [("Mei", "Can you book the room?"), ("Ravi", "Sure.")],
    [("Mei", "Shall we move it to Friday?"), ("Ravi", "Sure.")],
]
for case in sure_cases:
    case_text = chr(10).join(f"{s}: {t}" for s, t in case)
    case_turns, case_output, case_ends = run_pipeline(transcript=case_text)
    case_probs = label_probabilities(case_turns, trained)[-1]
    lead_in = case[0][1] if len(case) > 1 else "(no lead-in)"
    probs_str = ", ".join(f"{l} {p:.2f}" for l, p in sorted(zip(trained["labels"], case_probs), key=lambda x: -x[1]))
    print(f"{lead_in:28s} -> {probs_str}")


"Sure." alone could be anything (the three scores are close). After the request it's
confidently an action item -- right. After the proposal it's confidently... **discussion** --
**wrong**: a bare agreement to a proposal is a decision (`labelling_guide.md`). The model reads
context, same as the room does, but it doesn't always land on the right answer from it. That is
the honest version of the hook's closing line, and it's the lesson for block 9: context changes
the model's answer, but doesn't guarantee it's correct.


### YOUR TURN (Part B): the LSTM notebook update (Agenda block 8, Hands-on B, 20 min)
`new notebook = (how much to keep) × (old notebook) + (how much to add) × (the new note)`

`f` is the eraser, `i` is the pencil, `g` is the new note, `c_prev` is the old notebook. Write that one line.

In [ ]:
def lstm_cell_update(f, i, g, c_prev):
    # TODO: return the new notebook (the line above, in numpy)
    return None

hands_on.lstm_cell_update = lstm_cell_update
hands_on.FORGET_KNOB = 0          # try -6, -3, 0, 3, 6: negative erases more, positive keeps more
hands_on.part_b()

<details><summary>Stuck? (click to expand)</summary>

Verified by actually making each mistake and running it:

- **Bigger numbers than expected** -- you used `+` somewhere `*` should be
  (`f + c_prev` instead of `f * c_prev` keeps everything instead of erasing a share of it).
- **Close, but not quite** -- check the new note `g` is actually multiplied in
  (`i * g`, not just `i`).
- **`returned None`** -- the line still ends with `return None`; replace it with your formula.

</details>


Turn the knob to **-6** and **+6** and run the cell again. What happens to the first word, and to the trained model's score? Why do both extremes hurt it?

**All three settings, compared.** Run the knob at -6, 0 and +6 in one go, and look at both
numbers each time: the toy gap at word 12, and the trained model's score on the test meetings
(-6, -3, 0, +3, +6 give 10/45, 16/45, 35/45, 23/45 and 22/45). *(This sweep runs one of the five
models in the ensemble -- the same one `hands_on.py` uses -- while the "Sure." test above
averages all five; the two cells aren't measuring quite the same thing.)* "Keep a lot" (+6) is
**not** "better memory" -- the trained model learned its own eraser setting at 0, so pushing the
knob either way makes its labels worse:

- **-6 hurts** because the model erases everything, including the earlier line a reply like
  "Sure." needs (the hook's whole point) -- the toy gap dies at word 2.
- **+6 hurts** because the notebook barely erases anything: old content keeps piling up and
  swamps the new word each step, so the toy gap doesn't fade -- it actually **grows** from
  0.17 to 0.37, the opposite of fading, because every new note adds on top of notes that were
  never cleared.

The dial from above is this same forget gate: a knob of 0 sits at `sigmoid(0) = 0.5`, keeping
about half each step -- the 0.49 curve in the retention graph. A knob of +3 sits at
`sigmoid(3) = 0.953`, keeping about 95% -- the 0.95 curve. That's the link between the two
graphs: the retention illustration above is what this exact knob is tuning.


In [ ]:
for knob in (-6, 0, 6):
    hands_on.FORGET_KNOB = knob
    hands_on.part_b()
    print()
hands_on.FORGET_KNOB = 0   # leave the knob back at its trained setting


## 5. Meeting minutes (Agenda block 9, 15 min)
Now put it together: your two-speaker transcript goes through the trained LSTM, which labels every turn as a
**decision**, an **action item** or plain **discussion**, and the minutes are written from the labels.

In [ ]:
from IPython.display import Markdown, display
from lstm_model import run_pipeline
from meeting_minutes import label_turns_model, generate_minutes

turns, output, turn_ends = run_pipeline(transcript=transcript)
labels = label_turns_model(turns, output, turn_ends)
for (speaker, text), label in zip(turns, labels):
    print(f"{label['category']:<12} {speaker}: {text}")
print()
display(Markdown(generate_minutes(turns, labels)))


**Look closely.** Did it find both decisions? (It misses decisions stated as plain facts, such as "Venue is confirmed", and
a bare "Sounds good." after a proposal. That is where a model trained on few examples is weakest.)


### Your own meeting (Agenda block 9, part)
Write 5 lines from a meeting of your own -- a CCA, project or club -- one turn per line,
`Name: what they said`. Labelling is optional, but if you add one, label **every** line with
`act` (action item), `dec` (decision) or `dis` (discussion) at the start -- the model was tested
on 17 held-out meetings (clubs, projects, group work) and gets about 0.85-0.88 of turns right,
but its vocabulary is small, so unusual words for your topic are read as `<unk>` and ignored.
Names show up as unseen words too, for the same reason -- the model never actually reads them
(they are dropped before it sees the text), so this is expected and harmless.

The default below is the hook's own exchange, not an invented one -- so when you're asked which
earlier line the model should have remembered, there's a real answer.


In [ ]:
import os
import tempfile
from lstm_model import load_trained_model, label_probabilities
from pipeline import load_text, unknown_words
from IPython.display import Markdown, display

MY_MEETING = """
dis  Mei: Shall we move it to Friday?
dec  Ravi: Sure.
act  Mei: Can you book the room?
act  Ravi: Sure.
dis  Mei: Great, thanks.
"""

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8") as f:
    f.write(MY_MEETING)
    my_meeting_path = f.name
try:
    my_text, my_gold = load_text(my_meeting_path)
except ValueError as e:
    # Strip the temp-file path off the front of the message -- a participant doesn't need to
    # see where this notebook happened to save their typed text, just what to fix.
    message = str(e).replace(my_meeting_path, "").strip(" ,:")
    print("Couldn't read that meeting:", message)
    print("Hint: label every line or none of them, and write each line as 'Name: text'")
    print("(something after the colon).")
    my_text = None
finally:
    os.remove(my_meeting_path)

if my_text is not None:
    my_turns, my_output, my_ends = run_pipeline(transcript=my_text)
    my_labels = label_turns_model(my_turns, my_output, my_ends)
    my_probs = label_probabilities(my_turns, trained)

    for i, ((speaker, text), label, probs) in enumerate(zip(my_turns, my_labels, my_probs)):
        model_cat = label["category"]
        confidence = probs[trained["labels"].index(model_cat)]
        if my_gold:
            your_cat = my_gold[i]
            mark = "" if your_cat == model_cat else "  <- X"
            print(f"{your_cat:<12} (you)   {model_cat:<12} (model, {confidence:.0%} sure)   {speaker}: {text}{mark}")
        else:
            print(f"{model_cat:<12} (model, {confidence:.0%} sure)   {speaker}: {text}")
    if not my_gold:
        print()
        print("No labels given -- add act/dec/dis at the start of every line above and run again")
        print("to see where you and the model agree.")

    print()
    display(Markdown(generate_minutes(my_turns, my_labels)))

    unknown = unknown_words(my_turns)
    print()
    if unknown:
        print(f"Words the model never saw in training (read as <unk>): {', '.join(unknown)}")
    vocab_size = len(load_trained_model()["vocab"])
    print(f"The model knows {vocab_size} words; anything else is read as <unk>.")

    print()
    print("Pick one line it got wrong. Which EARLIER line should it have remembered?")
    print("(On the default example: the first \"Sure.\" is nearly 50-50 -- it needs the line")
    print("before it, \"Shall we move it to Friday?\", to be read as agreeing to a plan.)")


**Add a line before it, and watch it change.** Try it on the same knife-edge "Sure." by
putting one more line in front of the whole meeting:


In [ ]:
if my_text is not None:
    with_context = "Mei: Let's plan the showcase." + chr(10) + my_text
    with_context_turns, with_context_output, with_context_ends = run_pipeline(transcript=with_context)
    with_context_probs = label_probabilities(with_context_turns, trained)

    sure_count = 0
    sure_labels = ["first \"Sure.\" (after the proposal)", "second \"Sure.\" (after the request)"]
    for (speaker, text), probs in zip(with_context_turns, with_context_probs):
        if text == "Sure.":
            top = max(zip(trained["labels"], probs), key=lambda lp: lp[1])
            which = sure_labels[sure_count] if sure_count < len(sure_labels) else f"Sure. #{sure_count + 1}"
            print(f"{speaker}: {which:32s} -> {top[0]} ({top[1]:.0%})")
            sure_count += 1


One extra line of context earlier in the meeting is enough to flip the model's answer on
that first "Sure." -- from the default's near-50-50 miss to a (still narrow) correct
`decision`, 56% against 43%. **Caveat:** this is two examples on one knife-edge line, not a
general claim that "more context always helps" -- the model can just as easily be nudged the
wrong way by an earlier line, and most turns in this notebook aren't this close to begin with.


## 6. Wrap-up (Agenda block 10, 5 min)
**The whole journey, in one sentence:** sound becomes words, memory turns words into meaning,
and meaning becomes minutes.

**Three things this model honestly still gets wrong, seen in this notebook:**

1. A decision stated as a plain fact ("Venue is confirmed...") -- no "agree" or "let's" to catch
   it on.
2. Words it never saw in training -- read as `<unk>` and ignored.
3. A bare agreement to a proposal ("Sure." after "Shall we move it to Friday?") -- the model
   reads context the same way the room does, but doesn't always land on the right answer from it.

Take-home challenge: add your own 5 lines from "your own meeting" (section 5) and see whether a
retrained model would learn them -- the full code, including training, is with the facilitators
(not in this zip).


### Going further
* Change `MODEL` in section 1 to `small.en` or `medium.en`. How much does the transcript improve, and how much slower is it?
* Record a real two-person conversation (pause between speakers) and run it through `get_transcript`.
* Section 2 again with your own noise level or a different recording.